# LoRA train

Run `01_generate_training_data` first. DistilBERT + PEFT; fine on CPU, slow on big runs.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

_p = Path.cwd().resolve()
ROOT = next((a for a in [_p, *_p.parents] if (a / "pyproject.toml").is_file()), _p)
csv_path = ROOT / "notebooks" / "finetune_classifier" / "classification_train.csv"
df = pd.read_csv(csv_path)
labels = sorted(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df["label_id"] = df["label"].map(label2id)
train_df, val_df = train_test_split(df, test_size=0.1, stratify=df["label_id"], random_state=42)
model_name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(model_name)

def tok_batch(ex):
    enc = tok(ex["text"], truncation=True, padding="max_length", max_length=256)
    enc["labels"] = ex["labels"]
    return enc

train_ds = Dataset.from_pandas(train_df[["text", "label_id"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[["text", "label_id"]].reset_index(drop=True))
train_ds = train_ds.rename_column("label_id", "labels")
val_ds = val_ds.rename_column("label_id", "labels")
train_ds = train_ds.map(tok_batch, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tok_batch, batched=True, remove_columns=["text"])

base = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(labels), id2label=id2label, label2id=label2id
)
lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_lin", "v_lin"])
model = get_peft_model(base, lora)

out_dir = ROOT / "notebooks" / "finetune_classifier" / "lora_adapter"
out_dir.mkdir(parents=True, exist_ok=True)
args = TrainingArguments(
    output_dir=str(out_dir),
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
train_out = trainer.train()
model.save_pretrained(out_dir)
tok.save_pretrained(out_dir)
print("Training loss:", train_out.training_loss)
print("Saved to", out_dir)

In [ ]:
import matplotlib.pyplot as plt
h = trainer.state.log_history
losses = [x.get("loss") for x in h if "loss" in x]
if losses:
    plt.plot(losses)
    plt.title("Training loss (per log step)")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.show()